In [4]:
import sys
# Add project root to path
project_root = Path('/work')
sys.path.append(str(project_root))


In [7]:
#!/usr/bin/env python3
"""
Index Diagnostic Tool

This tool helps diagnose issues with item_key matching between the baseline index
and the resumen documents. It checks for missing keys, case sensitivity issues,
and other potential problems.
"""

import pickle
import random
from pathlib import Path
from typing import Dict, List, Set, Tuple, Any
from collections import Counter
import difflib

def diagnose_index_keys(index_path: str = "/work/sandbox/splade/baseline_indices",
                       vocab_path: str = "/work/sandbox/splade/vocabularies", 
                       data_path: str = "/work/data/processed",
                       file_base: str = "OEB",
                       use_filtered_vocab: bool = True,
                       max_features: int = 5000) -> Dict[str, Any]:
    """
    Comprehensive diagnostic of index vs resumen item_keys.
    
    Returns:
        Dictionary with diagnostic results
    """
    print("🔍 DIAGNOSING INDEX ITEM_KEY ISSUES")
    print("="*60)
    
    # Load the index
    from sandbox.splade.indexing.quick_builder import QuickSearchIndex, QuickVectorizer
    from sandbox.splade.core.vocabulary import VocabularyBuilder
    
    try:
        # Load vocabulary first
        vocab_suffix = "filtered" if use_filtered_vocab else "full"
        vocab_full_path = Path(vocab_path) / f"{file_base}_vocab_{vocab_suffix}_50000"
        
        print(f"Loading vocabulary from: {vocab_full_path}")
        vocab_builder = VocabularyBuilder.load_vocabulary(vocab_full_path)
        
        # Determine index path
        vocab_type = "filtered" if use_filtered_vocab else "full"
        features_suffix = f"_{max_features}" if max_features else ""
        index_name = f"{file_base}_baseline_index_{vocab_type}{features_suffix}"
        
        full_index_path = Path(index_path) / index_name
        
        if not full_index_path.exists():
            print(f"❌ Index not found at: {full_index_path}")
            print(f"🔧 Need to build index first!")
            return {"error": f"Index not found at {full_index_path}"}
        
        # Load index
        print(f"Loading index from: {full_index_path}")
        index = QuickSearchIndex.load(full_index_path, vocab_builder)
        
        print(f"✅ Index loaded successfully")
    except Exception as e:
        print(f"❌ Failed to load index: {e}")
        return {"error": str(e)}
    
    # Extract all keys from the index
    index_keys = set()
    for doc_info in index.documents:
        index_keys.add(doc_info['key'])
    
    print(f"📊 Index contains {len(index_keys)} documents")
    
    # Load resumen documents
    resumen_path = Path(data_path) / f"{file_base}_resumen.pkl"
    with open(resumen_path, 'rb') as f:
        resumen_docs = pickle.load(f)
    
    print(f"📊 Loaded {len(resumen_docs)} resumen documents")
    
    # Extract item_keys from resumen documents
    resumen_keys = set()
    resumen_key_extraction_issues = []
    
    for i, doc in enumerate(resumen_docs):
        # Use same extraction logic as in evaluation
        doc_key = None
        if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
            if 'item_key' in doc.metadata:
                doc_key = doc.metadata['item_key']
            elif 'doc_id' in doc.metadata:
                doc_key = doc.metadata['doc_id']
        
        if doc_key is None:
            if hasattr(doc, 'item_key'):
                doc_key = doc.item_key
            elif hasattr(doc, 'doc_id'):
                doc_key = doc.doc_id
            else:
                doc_key = f"resumen_{i}"
                resumen_key_extraction_issues.append(i)
        
        resumen_keys.add(doc_key)
    
    print(f"📊 Resumen documents have {len(resumen_keys)} unique keys")
    
    if resumen_key_extraction_issues:
        print(f"⚠️  {len(resumen_key_extraction_issues)} resumen docs had key extraction issues")
    
    # Compare the key sets
    print(f"\n🔍 KEY SET COMPARISON")
    print("-"*40)
    
    # Keys in resumen but not in index (the problem!)
    missing_in_index = resumen_keys - index_keys
    print(f"❌ Keys in resumen but NOT in index: {len(missing_in_index)}")
    
    # Keys in index but not in resumen  
    extra_in_index = index_keys - resumen_keys
    print(f"➕ Keys in index but NOT in resumen: {len(extra_in_index)}")
    
    # Keys present in both
    common_keys = resumen_keys & index_keys
    print(f"✅ Keys present in both: {len(common_keys)}")
    
    # Show some examples
    print(f"\n📋 EXAMPLE MISSING KEYS (first 10):")
    for key in list(missing_in_index)[:10]:
        print(f"  {key}")
    
    print(f"\n📋 EXAMPLE EXTRA INDEX KEYS (first 10):")
    for key in list(extra_in_index)[:10]:
        print(f"  {key}")
    
    print(f"\n📋 EXAMPLE COMMON KEYS (first 10):")
    for key in list(common_keys)[:10]:
        print(f"  {key}")
    
    # Check for case sensitivity issues
    print(f"\n🔍 CASE SENSITIVITY CHECK")
    print("-"*40)
    
    # Convert all to lowercase and check
    index_keys_lower = {key.lower() for key in index_keys}
    resumen_keys_lower = {key.lower() for key in resumen_keys}
    
    missing_lower = resumen_keys_lower - index_keys_lower
    print(f"Missing keys (case-insensitive): {len(missing_lower)}")
    
    if len(missing_lower) < len(missing_in_index):
        print(f"⚠️  Case sensitivity might be an issue!")
        print(f"  Original missing: {len(missing_in_index)}")
        print(f"  Lowercase missing: {len(missing_lower)}")
    
    # Check for near matches (typos, encoding issues)
    print(f"\n🔍 FUZZY MATCHING CHECK")
    print("-"*40)
    
    fuzzy_matches = []
    for missing_key in list(missing_in_index)[:5]:  # Check first 5 missing keys
        close_matches = difflib.get_close_matches(missing_key, index_keys, n=3, cutoff=0.8)
        if close_matches:
            fuzzy_matches.append((missing_key, close_matches))
    
    if fuzzy_matches:
        print(f"Found {len(fuzzy_matches)} potential fuzzy matches:")
        for missing, matches in fuzzy_matches:
            print(f"  '{missing}' -> {matches}")
    else:
        print(f"No close fuzzy matches found")
    
    # Check specific problematic key
    test_key = "OEB280chabb"
    print(f"\n🎯 SPECIFIC KEY CHECK: '{test_key}'")
    print("-"*40)
    
    in_resumen = test_key in resumen_keys
    in_index = test_key in index_keys
    
    print(f"In resumen: {in_resumen}")
    print(f"In index: {in_index}")
    
    if not in_index:
        # Look for similar keys
        similar = difflib.get_close_matches(test_key, index_keys, n=5, cutoff=0.7)
        print(f"Similar keys in index: {similar}")
    
    # Return diagnostic results
    results = {
        "index_key_count": len(index_keys),
        "resumen_key_count": len(resumen_keys),
        "missing_in_index": len(missing_in_index),
        "extra_in_index": len(extra_in_index), 
        "common_keys": len(common_keys),
        "case_sensitive_issues": len(missing_in_index) > len(missing_lower),
        "fuzzy_matches": fuzzy_matches,
        "missing_keys_sample": list(missing_in_index)[:20],
        "extra_keys_sample": list(extra_in_index)[:20],
        "resumen_extraction_issues": len(resumen_key_extraction_issues),
        "test_key_in_resumen": in_resumen,
        "test_key_in_index": in_index,
    }
    
    return results


def check_document_pairing(data_path: str = "/work/data/processed",
                          file_base: str = "OEB",
                          sample_size: int = 10) -> None:
    """
    Check if texto and resumen documents are properly paired.
    """
    print(f"\n🔍 CHECKING DOCUMENT PAIRING")
    print("="*40)
    
    # Load both document types
    texto_path = Path(data_path) / f"{file_base}_texto.pkl"
    resumen_path = Path(data_path) / f"{file_base}_resumen.pkl"
    
    with open(texto_path, 'rb') as f:
        texto_docs = pickle.load(f)
    
    with open(resumen_path, 'rb') as f:
        resumen_docs = pickle.load(f)
    
    print(f"Texto docs: {len(texto_docs)}")
    print(f"Resumen docs: {len(resumen_docs)}")
    
    # Extract keys from both
    def extract_key(doc, doc_type, index):
        if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
            if 'item_key' in doc.metadata:
                return doc.metadata['item_key']
            elif 'doc_id' in doc.metadata:
                return doc.metadata['doc_id']
        
        if hasattr(doc, 'item_key'):
            return doc.item_key
        elif hasattr(doc, 'doc_id'):
            return doc.doc_id
        else:
            return f"{doc_type}_{index}"
    
    texto_keys = set()
    resumen_keys = set()
    
    for i, doc in enumerate(texto_docs):
        key = extract_key(doc, "texto", i)
        texto_keys.add(key)
    
    for i, doc in enumerate(resumen_docs):
        key = extract_key(doc, "resumen", i)
        resumen_keys.add(key)
    
    print(f"Unique texto keys: {len(texto_keys)}")
    print(f"Unique resumen keys: {len(resumen_keys)}")
    
    # Check overlap
    common = texto_keys & resumen_keys
    texto_only = texto_keys - resumen_keys
    resumen_only = resumen_keys - texto_keys
    
    print(f"Common keys: {len(common)} ({len(common)/max(len(texto_keys), len(resumen_keys)):.1%})")
    print(f"Texto-only keys: {len(texto_only)}")
    print(f"Resumen-only keys: {len(resumen_only)}")
    
    # Show examples of each category
    if common:
        print(f"\nCommon keys (sample): {list(common)[:5]}")
    if texto_only:
        print(f"Texto-only (sample): {list(texto_only)[:5]}")
    if resumen_only:
        print(f"Resumen-only (sample): {list(resumen_only)[:5]}")
    
    # Check specific pairing
    sample_keys = random.sample(list(common), min(sample_size, len(common)))
    
    print(f"\n📋 SAMPLE DOCUMENT PAIRS:")
    for i, key in enumerate(sample_keys[:3]):
        # Find corresponding documents
        texto_doc = None
        resumen_doc = None
        
        for doc in texto_docs:
            if extract_key(doc, "texto", 0) == key:
                texto_doc = doc
                break
        
        for doc in resumen_docs:
            if extract_key(doc, "resumen", 0) == key:
                resumen_doc = doc
                break
        
        if texto_doc and resumen_doc:
            texto_text = texto_doc.text if hasattr(texto_doc, 'text') else str(texto_doc)
            resumen_text = resumen_doc.text if hasattr(resumen_doc, 'text') else str(resumen_doc)
            
            print(f"\nPair {i+1} - Key: {key}")
            print(f"  Texto:   {texto_text[:100]}...")
            print(f"  Resumen: {resumen_text[:100]}...")
        else:
            print(f"\nPair {i+1} - Key: {key} - ❌ MISSING DOCUMENT!")


def rebuild_index_with_diagnostics(file_base: str = "OEB",
                                  use_filtered_vocab: bool = True,
                                  max_features: int = 5000) -> None:
    """
    Rebuild the index with detailed diagnostics to see what's happening.
    """
    print(f"\n🔨 REBUILDING INDEX WITH DIAGNOSTICS")
    print("="*50)
    
    # First run key extraction test
    from sandbox.splade.indexing.quick_builder import test_item_key_extraction
    
    print(f"1. Testing key extraction on texto documents...")
    extraction_results = test_item_key_extraction()
    
    # Then rebuild index
    print(f"\n2. Rebuilding index...")
    from sandbox.splade.indexing.quick_builder import create_baseline_index
    
    index = create_baseline_index(
        vocab_path="/work/sandbox/splade/vocabularies",
        data_path="/work/data/processed",
        file_base=file_base,
        use_filtered_vocab=use_filtered_vocab,
        max_features=max_features,
        output_dir="/work/sandbox/splade/baseline_indices"
    )
    
    print(f"\n3. Running diagnostics on rebuilt index...")
    results = diagnose_index_keys(
        file_base=file_base,
        use_filtered_vocab=use_filtered_vocab,
        max_features=max_features
    )
    
    return results


def simple_index_check(file_base: str = "OEB",
                      use_filtered_vocab: bool = True, 
                      max_features: int = 5000) -> None:
    """
    Simple check to see what keys are actually in the current index.
    """
    print(f"🔍 SIMPLE INDEX KEY CHECK")
    print("="*40)
    
    # Build the expected index path
    vocab_type = "filtered" if use_filtered_vocab else "full"
    features_suffix = f"_{max_features}" if max_features else ""
    index_name = f"{file_base}_baseline_index_{vocab_type}{features_suffix}"
    
    index_path = Path("/work/sandbox/splade/baseline_indices") / index_name
    
    print(f"Looking for index at: {index_path}")
    print(f"Index exists: {index_path.exists()}")
    
    if not index_path.exists():
        print(f"❌ Index not found! Need to build it first.")
        print(f"💡 Run this to build the index:")
        print(f"   from sandbox.splade.indexing.quick_builder import create_baseline_index")
        print(f"   index = create_baseline_index(")
        print(f"       vocab_path='/work/sandbox/splade/vocabularies',")
        print(f"       data_path='/work/data/processed',")
        print(f"       file_base='{file_base}',")
        print(f"       use_filtered_vocab={use_filtered_vocab},")
        print(f"       max_features={max_features},")
        print(f"       output_dir='/work/sandbox/splade/baseline_indices'")
        print(f"   )")
        return
    
    try:
        # Load the index manually
        from sandbox.splade.indexing.quick_builder import QuickSearchIndex
        from sandbox.splade.core.vocabulary import VocabularyBuilder
        
        # Load vocabulary
        vocab_suffix = "filtered" if use_filtered_vocab else "full"
        vocab_path = Path("/work/sandbox/splade/vocabularies") / f"{file_base}_vocab_{vocab_suffix}_50000"
        vocab_builder = VocabularyBuilder.load_vocabulary(vocab_path)
        
        # Load index
        index = QuickSearchIndex.load(index_path, vocab_builder)
        
        print(f"✅ Index loaded successfully!")
        print(f"📊 Index contains {len(index.documents)} documents")
        
        # Show sample keys
        print(f"\n📋 Sample index keys:")
        for i, doc_info in enumerate(index.documents[:10]):
            print(f"  {i+1}. {doc_info['key']}")
        
        # Check for the problematic key
        test_key = "OEB280chabb"
        keys_in_index = {doc_info['key'] for doc_info in index.documents}
        
        print(f"\n🎯 Checking for problematic key: {test_key}")
        print(f"Key found in index: {test_key in keys_in_index}")
        
        if test_key not in keys_in_index:
            # Look for similar keys
            import difflib
            similar = difflib.get_close_matches(test_key, keys_in_index, n=5, cutoff=0.8)
            print(f"Similar keys: {similar}")
            
            # Check if any keys follow the expected pattern
            oeb_keys = [k for k in list(keys_in_index)[:20] if k.startswith('OEB')]
            uuid_keys = [k for k in list(keys_in_index)[:20] if '-' in k and len(k) > 20]
            
            print(f"\nKey patterns in index:")
            print(f"  OEB-style keys (sample): {oeb_keys[:5]}")
            print(f"  UUID-style keys (sample): {uuid_keys[:5]}")
            
            if uuid_keys and not oeb_keys:
                print(f"❌ PROBLEM: Index has UUID keys instead of OEB keys!")
                print(f"   This means the index was built with old key extraction logic.")
                print(f"   Need to rebuild the index.")
            elif oeb_keys:
                print(f"✅ Index has OEB-style keys, specific key just missing.")
        else:
            print(f"✅ Test key found in index!")
        
        return index
        
    except Exception as e:
        print(f"❌ Error loading index: {e}")
        import traceback
        traceback.print_exc()


# Quick diagnostic function for notebook use
def quick_diagnosis(file_base: str = "OEB",
                   max_features: int = 5000) -> Dict[str, Any]:
    """Quick diagnosis of the current index."""
    print("🚀 QUICK INDEX DIAGNOSIS")
    print("="*30)
    
    # Check document pairing first
    check_document_pairing(file_base=file_base)
    
    # Simple index check
    print(f"\n" + "="*50)
    index = simple_index_check(file_base=file_base, max_features=max_features)
    
    if index is None:
        return {"error": "Could not load index"}
    
    # If index loaded, do more detailed diagnosis
    try:
        results = diagnose_index_keys(file_base=file_base, max_features=max_features)
        
        # Summary
        print(f"\n📊 DIAGNOSIS SUMMARY:")
        print(f"  Index keys: {results.get('index_key_count', 'unknown')}")
        print(f"  Resumen keys: {results.get('resumen_key_count', 'unknown')}")
        print(f"  Missing in index: {results.get('missing_in_index', 'unknown')}")
        print(f"  Common keys: {results.get('common_keys', 'unknown')}")
        
        missing_count = results.get('missing_in_index', 0)
        if missing_count > 0:
            print(f"\n❌ PROBLEM DETECTED: {missing_count} resumen keys not found in index!")
            print(f"   This explains why target_text is empty for some queries.")
            print(f"   Solution: Rebuild index with fixed key extraction.")
        else:
            print(f"\n✅ All resumen keys found in index!")
        
        return results
    except Exception as e:
        print(f"❌ Detailed diagnosis failed: {e}")
        return {"error": str(e)}


if __name__ == "__main__":
    # Run quick diagnosis
    results = quick_diagnosis()
    
    # If there are issues, suggest rebuilding
    if results.get('missing_in_index', 0) > 0:
        print(f"\n🔧 RECOMMENDED ACTION:")
        print(f"   Run: rebuild_index_with_diagnostics()")
        print(f"   This will rebuild the index with proper key extraction.")

🚀 QUICK INDEX DIAGNOSIS

🔍 CHECKING DOCUMENT PAIRING
Texto docs: 47514
Resumen docs: 47514
Unique texto keys: 47514
Unique resumen keys: 47514
Common keys: 47514 (100.0%)
Texto-only keys: 0
Resumen-only keys: 0

Common keys (sample): ['OEB280ecbcc', 'OEB230ehdac', 'OEB280jabdb', 'OEB280hgdca', 'OEB020deebc']

📋 SAMPLE DOCUMENT PAIRS:

Pair 1 - Key: OEB040fcebb
  Texto:   Canalización hormigonada de  6  tubos de polietileno libre de halógenos de 160 mm de diámetro en ter...
  Resumen: Canalización hormigonada  6  T, polietileno libre de halógenos de 160 mm, rocoso. (Cualquier frana h...

Pair 2 - Key: OEB040hdbcc
  Texto:   Canalización hormigonada de  12  tubos de polietileno libre de halógenos de 160 mm de diámetro en cr...
  Resumen: Canalización hormigonada  12  T, polietileno libre de halógenos de 160 mm, en cruce de carretera. (N...

Pair 3 - Key: OEB290jacba
  Texto:   Canalización hormigonada de  18  tubos de polietileno libre de halógenos de 50 mm de diámetro en cua...
  Resume

In [8]:
#!/usr/bin/env python3
"""
Evaluation Logic Debug Tool

This tool debugs the evaluation process step-by-step to find why
target_text is empty even when the index contains the correct keys.
"""

import pickle
import random
from pathlib import Path
from typing import Dict, List, Any

def debug_evaluation_logic(index_path: str = "/work/sandbox/splade/baseline_indices",
                          vocab_path: str = "/work/sandbox/splade/vocabularies", 
                          data_path: str = "/work/data/processed",
                          file_base: str = "OEB",
                          use_filtered_vocab: bool = True,
                          max_features: int = 5000,
                          test_key: str = "OEB280chabb") -> None:
    """
    Debug the evaluation logic step by step.
    """
    print("🐛 DEBUGGING EVALUATION LOGIC")
    print("="*50)
    
    # Load the index
    from sandbox.splade.indexing.quick_builder import QuickSearchIndex
    from sandbox.splade.core.vocabulary import VocabularyBuilder
    
    # Load vocabulary
    vocab_suffix = "filtered" if use_filtered_vocab else "full"
    vocab_full_path = Path(vocab_path) / f"{file_base}_vocab_{vocab_suffix}_50000"
    vocab_builder = VocabularyBuilder.load_vocabulary(vocab_full_path)
    
    # Load index
    vocab_type = "filtered" if use_filtered_vocab else "full"
    features_suffix = f"_{max_features}" if max_features else ""
    index_name = f"{file_base}_baseline_index_{vocab_type}{features_suffix}"
    full_index_path = Path(index_path) / index_name
    
    index = QuickSearchIndex.load(full_index_path, vocab_builder)
    print(f"✅ Index loaded: {len(index.documents)} documents")
    
    # Load resumen documents
    resumen_path = Path(data_path) / f"{file_base}_resumen.pkl"
    with open(resumen_path, 'rb') as f:
        resumen_docs = pickle.load(f)
    
    print(f"✅ Resumen docs loaded: {len(resumen_docs)}")
    
    # Find the problematic resumen document
    target_resumen_doc = None
    for doc in resumen_docs:
        # Extract key using same logic as evaluation
        doc_key = None
        if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
            if 'item_key' in doc.metadata:
                doc_key = doc.metadata['item_key']
            elif 'doc_id' in doc.metadata:
                doc_key = doc.metadata['doc_id']
        
        if doc_key is None:
            if hasattr(doc, 'item_key'):
                doc_key = doc.item_key
            elif hasattr(doc, 'doc_id'):
                doc_key = doc.doc_id
        
        if doc_key == test_key:
            target_resumen_doc = doc
            break
    
    if target_resumen_doc is None:
        print(f"❌ Could not find resumen document with key: {test_key}")
        return
    
    print(f"✅ Found target resumen document")
    
    # Debug step by step
    print(f"\n🔍 STEP-BY-STEP DEBUGGING")
    print("-"*40)
    
    # Step 1: Extract query info
    query_text = target_resumen_doc.text if hasattr(target_resumen_doc, 'text') else str(target_resumen_doc)
    
    print(f"Step 1 - Query extraction:")
    print(f"  Query key: {test_key}")
    print(f"  Query text: {query_text[:100]}...")
    
    # Step 2: Search index  
    print(f"\nStep 2 - Index search:")
    search_results = index.search(query_text, top_k=10, min_score=0.0)
    print(f"  Search returned: {len(search_results)} results")
    
    if search_results:
        print(f"  Top result key: {search_results[0].item_key}")
        print(f"  Top result score: {search_results[0].score:.6f}")
        print(f"  Top result text: {search_results[0].text[:100]}...")
    
    # Step 3: Look for target in results
    print(f"\nStep 3 - Target matching:")
    target_found = False
    target_position = -1
    target_score = 0.0
    target_text = ""
    
    for pos, result in enumerate(search_results):
        print(f"  Result {pos+1}: {result.item_key} (score: {result.score:.6f})")
        if result.item_key == test_key:
            target_found = True
            target_position = pos + 1
            target_score = result.score
            target_text = result.text
            print(f"    ⭐ TARGET FOUND at position {pos+1}!")
    
    print(f"\nStep 4 - Target summary:")
    print(f"  Target found: {target_found}")
    print(f"  Target position: {target_position}")
    print(f"  Target score: {target_score}")
    print(f"  Target text length: {len(target_text)}")
    
    # Step 5: Simulate evaluation result creation
    print(f"\nStep 5 - Evaluation result creation:")
    eval_result = {
        'resumen_item_key': test_key,
        'resumen_text': query_text,
        'target_text': target_text,
        'target_score': target_score,
        'texto_results': [
            {
                'item_key': result.item_key,
                'text': result.text,
                'score': result.score
            }
            for result in search_results
        ]
    }
    
    print(f"  Created eval result:")
    print(f"    resumen_item_key: {eval_result['resumen_item_key']}")
    print(f"    target_text length: {len(eval_result['target_text'])}")
    print(f"    target_score: {eval_result['target_score']}")
    print(f"    num texto_results: {len(eval_result['texto_results'])}")
    
    # Check if this matches what we expect
    if target_found:
        print(f"\n✅ EVALUATION LOGIC WORKING CORRECTLY!")
        print(f"   Target should be found with score {target_score:.6f}")
    else:
        print(f"\n❌ PROBLEM IN EVALUATION LOGIC!")
        print(f"   Target key {test_key} not found in search results")
        print(f"   But we know it exists in the index...")
        
        # Debug further - check if key exists in index documents
        index_keys = {doc_info['key'] for doc_info in index.documents}
        if test_key in index_keys:
            print(f"   🤔 Key exists in index.documents but not in search results")
            print(f"   This suggests a search/vectorization issue")
        else:
            print(f"   🤔 Key doesn't exist in index.documents")
            print(f"   This suggests an indexing issue")


def test_multiple_problematic_cases(num_tests: int = 5) -> None:
    """
    Test multiple cases that showed empty target_text.
    """
    print(f"🧪 TESTING MULTIPLE PROBLEMATIC CASES")
    print("="*50)
    
    # Load index and resumen docs
    from sandbox.splade.indexing.quick_builder import QuickSearchIndex
    from sandbox.splade.core.vocabulary import VocabularyBuilder
    
    # Load vocabulary
    vocab_path = Path("/work/sandbox/splade/vocabularies") / "OEB_vocab_filtered_50000"
    vocab_builder = VocabularyBuilder.load_vocabulary(vocab_path)
    
    # Load index
    index_path = Path("/work/sandbox/splade/baseline_indices") / "OEB_baseline_index_filtered_5000"
    index = QuickSearchIndex.load(index_path, vocab_builder)
    
    # Load resumen documents
    resumen_path = Path("/work/data/processed") / "OEB_resumen.pkl"
    with open(resumen_path, 'rb') as f:
        resumen_docs = pickle.load(f)
    
    print(f"✅ Loaded index and {len(resumen_docs)} resumen docs")
    
    # Test random samples
    sample_docs = random.sample(resumen_docs, num_tests)
    
    for i, doc in enumerate(sample_docs):
        print(f"\n{'='*20} TEST {i+1} {'='*20}")
        
        # Extract key and text
        doc_key = None
        if hasattr(doc, 'metadata') and isinstance(doc.metadata, dict):
            if 'item_key' in doc.metadata:
                doc_key = doc.metadata['item_key']
        
        if doc_key is None:
            print(f"❌ Could not extract key from document {i}")
            continue
        
        query_text = doc.text if hasattr(doc, 'text') else str(doc)
        
        print(f"Query key: {doc_key}")
        print(f"Query text: {query_text[:80]}...")
        
        # Search
        results = index.search(query_text, top_k=5, min_score=0.0)
        
        # Check for target
        target_found = False
        for j, result in enumerate(results):
            if result.item_key == doc_key:
                target_found = True
                print(f"✅ Target found at position {j+1} with score {result.score:.6f}")
                break
        
        if not target_found:
            print(f"❌ Target NOT found in top 5 results!")
            print(f"   Top results:")
            for j, result in enumerate(results[:3]):
                print(f"     {j+1}. {result.item_key} (score: {result.score:.6f})")


def debug_specific_case() -> None:
    """Debug the exact case from the user's output."""
    print(f"🎯 DEBUGGING SPECIFIC CASE: OEB280chabb")
    print("="*50)
    
    debug_evaluation_logic(
        test_key="OEB280chabb",
        max_features=5000  # Note: user was using 5000, not 10000
    )


if __name__ == "__main__":
    # Debug the specific problematic case
    debug_specific_case()
    
    print(f"\n" + "="*60)
    
    # Test multiple cases
    test_multiple_problematic_cases(num_tests=3)

🎯 DEBUGGING SPECIFIC CASE: OEB280chabb
🐛 DEBUGGING EVALUATION LOGIC
Vocabulary loaded from /work/sandbox/splade/vocabularies/OEB_vocab_filtered_50000
Vocabulary size: 333
QuickVectorizer initialized:
  Vocabulary size: 328
  Max features: 5000
Vectorizer loaded from /work/sandbox/splade/baseline_indices/OEB_baseline_index_filtered_5000
QuickSearchIndex initialized
Search index loaded from /work/sandbox/splade/baseline_indices/OEB_baseline_index_filtered_5000
✅ Index loaded: 47514 documents
✅ Resumen docs loaded: 47514
✅ Found target resumen document

🔍 STEP-BY-STEP DEBUGGING
----------------------------------------
Step 1 - Query extraction:
  Query key: OEB280chabb
  Query text: Canalización hormigonada  3  T, polietileno libre de halógenos de 40 mm, con topo. (Diurno/3 <== i <...

Step 2 - Index search:
Searching with 1 queries...
✅ Search complete:
  Average results per query: 10.0
  Search returned: 10 results
  Top result key: OEB280cdcbb
  Top result score: 0.673764
  Top result 